In [1]:
from leagueScripts import PlayTypeLeagueAverage
from nba_api.stats.library.parameters import PlayType
from nba_api.stats.endpoints import SynergyPlayTypes
import pandas as pd

playtypes_df = []
for playtype in PlayTypeLeagueAverage.get_playtype_classes_names():
    _playtype_df = SynergyPlayTypes(play_type_nullable=getattr(PlayType, playtype), type_grouping_nullable='offensive', player_or_team_abbreviation='P').synergy_play_type.get_data_frame()
    playtypes_df.append(_playtype_df)
df = pd.concat(playtypes_df)
df = df[['PLAYER_NAME', 'TEAM_NAME', 'PLAY_TYPE', 'PTS', 'POSS', 'POSS_PCT']]

# Define a custom aggregation function
def custom_agg(group):
    # Calculate the combined team names
    combined_team_names = ', '.join(group['TEAM_NAME'].unique())
    
    # Calculate the sum of PTS and POSS
    pts_sum = group['PTS'].sum()
    poss_sum = group['POSS'].sum()
    
    # Calculate the correct POSS_PCT
    poss_pct_numerator = poss_sum
    poss_pct_denominator = (group['POSS'] / group['POSS_PCT']).sum()
    poss_pct = poss_pct_numerator / poss_pct_denominator if poss_pct_denominator != 0 else 0
    
    # Return as a Series
    return pd.Series({
        'TEAM_NAME': combined_team_names,
        'PTS': pts_sum,
        'POSS': poss_sum,
        'POSS_PCT': poss_pct
    })

# Apply the custom aggregation for players with multiple teams
df = df.groupby(['PLAYER_NAME', 'PLAY_TYPE']).apply(custom_agg, include_groups=False)
df['PPP'] = df['PTS'] / df['POSS']
# Calculate percentiles
df['PERCENTILE_FREQ%'] = df.groupby('PLAY_TYPE')['POSS_PCT'].rank(pct=True) * 100

df

TEAM_NAME  PTS  POSS  POSS_PCT  \
PLAYER_NAME     PLAY_TYPE                                                  
AJ Green        Handoff             Milwaukee Bucks   33    23     0.110   
                OffScreen           Milwaukee Bucks   19    20     0.095   
                PRBallHandler       Milwaukee Bucks    9    14     0.067   
                PRRollMan           Milwaukee Bucks   17    14     0.067   
                Spotup              Milwaukee Bucks  111    90     0.429   
...                                             ...  ...   ...       ...   
Zion Williamson PRBallHandler  New Orleans Pelicans  249   264     0.178   
                PRRollMan      New Orleans Pelicans   22    24     0.016   
                Postup         New Orleans Pelicans  168   188     0.127   
                Spotup         New Orleans Pelicans   89    91     0.061   
                Transition     New Orleans Pelicans  282   204     0.138   

                                    PPP  PERCENTILE_FREQ%  
PLAYER_NAME     PLAY_TYPE                                  
AJ Green        Handoff        1.434783         90.449438  
                OffScreen      0.950000         86.197917  
                PRBallHandler  0.642857         20.848057  
                PRRollMan      1.214286         52.500000  
                Spotup         1.233333         83.569405  
...                                 ...               ...  
Zion Williamson PRBallHandler  0.943182         55.123675  
                PRRollMan      0.916667         10.625000  
                Postup         0.893617         83.333333  
                Spotup         0.978022          4.532578  
                Transition     1.382353         24.852941  

[2929 rows x 6 columns]

In [2]:
# Pivot the dataframe
df_pivot = df.pivot_table(index=['PLAYER_NAME'], 
                          columns='PLAY_TYPE', 
                          values=['PPP', 'POSS_PCT', 'PERCENTILE_FREQ%'], 
                          aggfunc='first', 
                          fill_value=0)

# Flatten the columns
# df_pivot.columns = [f'{i}_{j}' for i, j in df_pivot.columns]
df_pivot = df_pivot.rename(columns={'PLAYER_NAME_': 'PLAYER_NAME', 'TEAM_NAME_': 'TEAM_NAME'})

df_ppp_for_plot = df_pivot['PPP']
df_frequency_for_plot = df_pivot['POSS_PCT']
df_percentile_frequency_for_plot = df_pivot['PERCENTILE_FREQ%']
metrics = df_ppp_for_plot.columns.tolist()

In [3]:
df_ppp_for_plot

PLAY_TYPE,Cut,Handoff,Isolation,Misc,OffRebound,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup,Transition
PLAYER_NAME,,,,,,,,,,,
AJ Green,0.000000,1.434783,0.000000,0.000000,0.000000,0.950000,0.642857,1.214286,0.000000,1.233333,0.000000
Aaron Gordon,1.529762,0.800000,0.637500,0.477273,0.000000,0.000000,0.608696,1.339286,1.037037,1.071429,1.069892
Aaron Holiday,0.000000,0.826087,1.030303,0.000000,0.846154,0.894737,1.027397,0.000000,0.000000,1.131250,1.014493
Aaron Nesmith,1.137931,1.184211,0.000000,0.500000,1.285714,1.250000,0.000000,1.480000,0.000000,1.161616,1.274611
Aaron Wiggins,1.301587,1.062500,0.000000,0.454545,1.037037,0.000000,1.055556,1.480000,0.000000,1.260274,1.139130
...,...,...,...,...,...,...,...,...,...,...,...
Zach Collins,1.355556,0.000000,1.045455,0.252874,1.000000,0.000000,0.000000,0.991935,0.975610,0.884615,1.049180
Zach LaVine,0.000000,1.080000,0.833333,0.846154,0.000000,0.875000,0.941558,0.000000,0.000000,1.037975,1.201923
Zavier Simpson,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.666667,0.000000,0.000000,0.000000,0.818182


In [4]:
df_frequency_for_plot

PLAY_TYPE,Cut,Handoff,Isolation,Misc,OffRebound,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup,Transition
PLAYER_NAME,,,,,,,,,,,
AJ Green,0.000,0.110,0.000,0.000,0.000,0.095,0.067,0.067,0.000,0.429,0.000
Aaron Gordon,0.179,0.021,0.085,0.047,0.000,0.000,0.049,0.060,0.115,0.105,0.199
Aaron Holiday,0.000,0.047,0.067,0.000,0.026,0.038,0.296,0.000,0.000,0.324,0.140
Aaron Nesmith,0.039,0.051,0.000,0.054,0.038,0.059,0.000,0.034,0.000,0.399,0.259
Aaron Wiggins,0.136,0.034,0.000,0.047,0.058,0.000,0.078,0.054,0.000,0.315,0.248
...,...,...,...,...,...,...,...,...,...,...,...
Zach Collins,0.110,0.000,0.027,0.106,0.083,0.000,0.000,0.303,0.150,0.127,0.074
Zach LaVine,0.000,0.053,0.127,0.028,0.000,0.034,0.327,0.000,0.000,0.168,0.221
Zavier Simpson,0.000,0.000,0.000,0.000,0.000,0.000,0.409,0.000,0.000,0.000,0.167


In [5]:
df_percentile_frequency_for_plot

PLAY_TYPE,Cut,Handoff,Isolation,Misc,OffRebound,OffScreen,PRBallHandler,PRRollMan,Postup,Spotup,Transition
PLAYER_NAME,,,,,,,,,,,
AJ Green,0.000000,90.449438,0.000000,0.000000,0.000000,86.197917,20.848057,52.500000,0.000000,83.569405,0.000000
Aaron Gordon,83.274021,8.426966,70.512821,38.424437,0.000000,0.000000,13.780919,49.791667,79.166667,10.623229,65.000000
Aaron Holiday,0.000000,37.265918,57.264957,0.000000,21.654930,48.437500,76.855124,0.000000,0.000000,57.648725,25.441176
Aaron Nesmith,24.911032,44.194757,0.000000,50.803859,37.323944,68.229167,0.000000,33.333333,0.000000,75.920680,92.352941
Aaron Wiggins,77.580071,20.411985,0.000000,38.424437,52.640845,0.000000,23.851590,46.041667,0.000000,54.107649,90.735294
...,...,...,...,...,...,...,...,...,...,...,...
Zach Collins,73.487544,0.000000,15.811966,92.443730,65.492958,0.000000,0.000000,98.750000,90.972222,13.881020,4.705882
Zach LaVine,0.000000,48.127341,85.897436,5.144695,0.000000,41.666667,83.568905,0.000000,0.000000,18.838527,77.352941
Zavier Simpson,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,94.699647,0.000000,0.000000,0.000000,43.823529


In [15]:
from leagueScripts import NBALeague

league_object = NBALeague.get_cached_league_object()

top_scorers = sorted(league_object.players_on_teams_objects_list, key=lambda x: x.stats_df['PTS'].sum().item() if not x.stats_df.empty else 0, reverse=True)[:100]
players_to_plot = [player.player_info['DISPLAY_FIRST_LAST'].item() for player in top_scorers]
title = f"{', '.join(players_to_plot) if len(players_to_plot) <5 else 'Players'} play type stats"

In [17]:
import numpy as np
import plotly.graph_objects as go

# Create a new Figure
fig = go.Figure()

# Add averages
fig.add_trace(go.Scatterpolar(
        r=df_ppp_for_plot.replace(0, np.NaN).mean(),
        theta=metrics,
        name="Average",
        line=dict(color='white'),  # Set the color for the average trace
        showlegend=False  # Make the Average trace always visible and not part of the legend
    ))

# Plot each player in a separate chart
for player_name in players_to_plot:
    player_ppp_df = df_ppp_for_plot.loc[player_name]
    player_frequency_df = df_frequency_for_plot.loc[player_name]
    player_percentile_frequency_df = df_percentile_frequency_for_plot.loc[player_name]

    fig.add_trace(go.Scatterpolar(
        r=player_ppp_df.values,
        theta=metrics,
        fill='toself',
        name=player_name,
        marker=dict(
            size=player_percentile_frequency_df.values
        ),
        customdata=player_frequency_df.values * 100,  # Adding custom data
        hovertemplate='<b>%{theta}</b><br>PPP: %{r}<br>Frequency: %{customdata}%<extra></extra>'
    
    ))

# Update layout with increased width and height
fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 2]
        )),
    showlegend=True,
    title=title,
    width=1500,
    height=1000,
)

# Show the plot
fig.show()
